# Bias Ridge TF-IDF - Colab

5클래스 편향도 연속점수 예측용 TF-IDF + Ridge 회귀 노트북입니다.

- `PROJECT_ROOT`는 노트북이 자동 탐색합니다.
- 회귀 노트북은 5클래스 split을 자동 생성합니다.


In [ ]:
!pip -q install pandas scikit-learn joblib xgboost


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
SPLIT_NAME = 'splits_label1_5class'
MODEL_SUBDIR = 'regression/bias_ridge_tfidf/models'
from pathlib import Path
import subprocess
import sys

import joblib
import numpy as np
import pandas as pd

def _find_project_root():
    candidates = []
    cwd = Path.cwd()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        Path('/content/drive/MyDrive/sw_project/ai_features/political_bias_analysis'),
        Path('/content/drive/MyDrive/ai_features/political_bias_analysis'),
        Path('/content/sw_project/ai_features/political_bias_analysis'),
        Path('/content/ai_features/political_bias_analysis'),
        Path('/content/political_bias_analysis'),
    ])
    for path in candidates:
        if (path / 'prepare_label1_splits.py').exists():
            return path
    for base in [Path('/content'), Path('/content/drive'), Path('/content/drive/MyDrive')]:
        if base.exists():
            for match in base.rglob('prepare_label1_splits.py'):
                return match.parent
    raise FileNotFoundError('Could not find political_bias_analysis project root. Mount the repo or update _find_project_root().')

PROJECT_ROOT = _find_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
SPLIT_DIR = DATA_DIR / SPLIT_NAME
MODEL_DIR = PROJECT_ROOT / MODEL_SUBDIR
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = SPLIT_DIR / 'train.csv'
VALID_CSV = SPLIT_DIR / 'valid.csv'
TEST_CSV = SPLIT_DIR / 'test.csv'

print('project_root =', PROJECT_ROOT)
print('split_dir =', SPLIT_DIR)
print('model_dir =', MODEL_DIR)


In [ ]:
import subprocess
import time


def _log(msg):
    print(f'[split] {msg}')


def ensure_splits():
    split_start = time.perf_counter()
    raw_dir = DATA_DIR / 'original_data'
    train_source = raw_dir / 'complete_train.csv'
    test_source = raw_dir / 'complete_test.csv'

    _log(f'train_source = {train_source.exists()} | {train_source}')
    _log(f'test_source = {test_source.exists()} | {test_source}')

    if TRAIN_CSV.exists() and VALID_CSV.exists() and TEST_CSV.exists():
        _log('split files already exist; skipping creation')
        print(f'[split] elapsed = {time.perf_counter() - split_start:.2f}s')
        return

    if not train_source.exists() or not test_source.exists():
        raise FileNotFoundError('Raw CSV files are missing. Need data/original_data/complete_train.csv and complete_test.csv.')

    result = subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / 'prepare_label1_splits.py'),
            '--label-mode', 'three_class' if SPLIT_NAME.endswith('3class') else 'five_class',
            '--train-csv', str(train_source),
            '--test-csv', str(test_source),
            '--out-dir', str(SPLIT_DIR),
        ],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    print(result.stderr)
    result.check_returncode()
    _log('split generation completed')
    print(f'[split] elapsed = {time.perf_counter() - split_start:.2f}s')


ensure_splits()


In [ ]:
import time

from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error

import json

def eval_regression(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    rounded_pred = np.clip(np.rint(y_pred), 1, 5).astype(int)
    rounded_true = np.clip(np.rint(y_true), 1, 5).astype(int)
    acc = accuracy_score(rounded_true, rounded_pred)
    macro_f1 = f1_score(rounded_true, rounded_pred, average='macro')
    print(f'[{name}] mae = {mae:.4f}')
    print(f'[{name}] rmse = {rmse:.4f}')
    print(f'[{name}] rounded_accuracy = {acc:.4f}')
    print(f'[{name}] rounded_macro_f1 = {macro_f1:.4f}')
    return {
        'mae': float(mae),
        'rmse': float(rmse),
        'rounded_accuracy': float(acc),
        'rounded_macro_f1': float(macro_f1),
    }
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.pipeline import FeatureUnion, Pipeline

def _log(msg):
    print(f'[train] {msg}')

def load_split(path):
    df = pd.read_csv(path)
    return df['text'].fillna('').astype(str), df['label'].astype(float).to_numpy()

def eval_regression(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    rounded_pred = np.clip(np.rint(y_pred), 1, 5).astype(int)
    rounded_true = np.clip(np.rint(y_true), 1, 5).astype(int)
    acc = accuracy_score(rounded_true, rounded_pred)
    macro_f1 = f1_score(rounded_true, rounded_pred, average='macro')
    print(f'[{name}] mae = {mae:.4f}')
    print(f'[{name}] rmse = {rmse:.4f}')
    print(f'[{name}] rounded_accuracy = {acc:.4f}')
    print(f'[{name}] rounded_macro_f1 = {macro_f1:.4f}')

def build_model():
    return Pipeline([
        ('features', FeatureUnion([
            ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, max_df=0.98, sublinear_tf=True)),
            ('char', TfidfVectorizer(analyzer='char', ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
        ])),
        ('reg', Ridge(alpha=1.0)),
    ])

train_start = time.perf_counter()
_log('loading split data')
load_start = time.perf_counter()
X_train, y_train = load_split(TRAIN_CSV)
X_valid, y_valid = load_split(VALID_CSV)
X_test, y_test = load_split(TEST_CSV)
_log(f'loaded data in {time.perf_counter() - load_start:.2f}s')

model = build_model()
_log('fitting model')
fit_start = time.perf_counter()
model.fit(X_train, y_train)
_log(f'fit completed in {time.perf_counter() - fit_start:.2f}s')

_log('running evaluation')
valid_pred = model.predict(X_valid)
test_pred = model.predict(X_test)

valid_metrics = eval_regression('validation', y_valid, valid_pred)
test_metrics = eval_regression('test', y_test, test_pred)

result = {
    'model_name': 'bias_ridge_tfidf',
    'split_name': SPLIT_NAME,
    'valid': valid_metrics,
    'test': test_metrics,
}
result_path = MODEL_DIR / 'bias_ridge_tfidf_metrics.json'
with open(result_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print('saved metrics to', result_path)

_log('saving model')
joblib.dump(model, MODEL_DIR / 'bias_ridge_tfidf_colab.joblib')
print('saved to', MODEL_DIR / 'bias_ridge_tfidf_colab.joblib')
_log(f'total elapsed = {time.perf_counter() - train_start:.2f}s')

